# Collecte Api des données climatiques et de géolocalisation

In [5]:
import pandas as pd
import requests 
import time
import plotly.express as px
from datetime import datetime, timedelta


In [6]:
# filtre initial des villes françaises
df_ville = ["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

### Bloc villes données géographique

In [ ]:
# définitiion du Header et de l'url api endpoint -> /search
api_nomina_url ='https://nominatim.openstreetmap.org/search?q='
headers = {'User-Agent': 'projet_kayak'}
api_ville_adresse = []

# pour chaque ville on fait un appel api 'get' à nominatim, on récupère le nom et les coordonnées 
for compteur,ville in enumerate(df_ville):
    id_ville = compteur + 1
    response = requests.get(api_nomina_url+ville+'&format=json&limit=1',headers=headers)


    if response.status_code == 200:
        data = response.json()[0]
        api_ville_adresse.append({
            "id": id_ville,
            "ville": ville,
            "latitude": float(data["lat"]),
            "longitude": float(data["lon"])
        })
        
    else:
        print(f'problème avec la ville :{ville}')

    
    time.sleep(1)


df_ville_location = pd.DataFrame(api_ville_adresse)


In [8]:
# contrôle des données récupérées
df_ville_location.describe(include='all')

,id,ville,latitude,longitude
count,35.000000,35,35.000000,35.000000
unique,NaN,35,NaN,NaN
top,NaN,Mont Saint Michel,NaN,NaN
freq,NaN,1,NaN,NaN
mean,18.000000,NaN,45.840997,3.396134
std,10.246951,NaN,2.590086,2.954175
min,1.000000,NaN,42.525050,-2.026041
25%,9.500000,NaN,43.512178,1.380777
50%,18.000000,NaN,45.187560,4.360069
75%,26.500000,NaN,48.416998,5.637707


### Bloc villes données climatique

In [9]:
# données météo
# utilisation d'une api gratuite et moins restrictive et sans clé api -> api.open-meteo

ville_meteo = []

for index, row in df_ville_location.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    
    url_weather = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_max,rain_sum,relative_humidity_2m_max&timezone=Europe%2FParis"
    
    response = requests.get(url_weather)
    
    # si la réponse est ok
    if response.status_code == 200:
        data = response.json()
        
        # prévision à 7 jours
        daily = data.get("daily", {})  

        # température pluie humidité 
        temps = daily.get("temperature_2m_max", [])
        pluie = daily.get("rain_sum", [])
        humidite = daily.get("relative_humidity_2m_max", [])
        
        # pour moi le 'meilleur temps' est celui qui est le moins chaud et sec
            # ce sera donc le score le plus bas pour la somme des moyennes 
        
        beau_temps = round( (sum(temps)/7) + (sum(pluie)/7) + (sum(humidite)/7) , 2 )

        ville_meteo.append({
            "id": row["id"],
            "ville": row["ville"],
            "latitude": lat,
            "longitude": lon,
            "beau_temps": beau_temps,
            "temperature": round((sum(temps)/7),2),
            "pluie": round((sum(pluie)/7),2),
            "humidite": round((sum(humidite)/7),2)

        })

       
    else:
        print(f'problème avec la ville :{row[ville]}')
    
    #time.sleep(1)

df_meteo_complet = pd.DataFrame(ville_meteo)

In [10]:
# plus le beau_temps est faible plus il fait fraic et sec
df_meteo_complet.sort_values('beau_temps')

,id,ville,latitude,longitude,beau_temps,temperature,pluie,humidite
20,21,Marseille,43.296399,5.377789,95.94,32.94,0.00,63.00
18,19,Bormes les Mimosas,43.150697,6.341928,97.59,34.73,0.00,62.86
19,20,Cassis,43.214036,5.539632,98.93,33.21,0.00,65.71
9,10,Chateau du Haut Koenigsbourg,48.249382,7.343941,101.26,30.79,0.04,70.43
5,6,Paris,48.858890,2.320041,101.94,31.97,0.26,69.71
16,17,Lyon,45.757814,4.832011,102.11,35.60,0.23,66.29
34,35,La Rochelle,46.159732,-1.151595,103.79,30.17,0.04,73.57
13,14,Dijon,47.321581,5.041470,104.01,34.44,0.00,69.57
28,29,Carcassonne,43.213036,2.349107,104.87,34.46,0.13,70.29
11,12,Eguisheim,48.044797,7.307962,105.44,35.16,0.00,70.29


In [11]:
# ajout de la période de recherche

# de ajourd'hui
date_jour = datetime.now().strftime('%d/%m/%Y')
# a dans 7 jours
date_dans_7_jours = (datetime.now() + timedelta(days=7)).strftime('%d/%m/%Y')

periode_texte = f"du {date_jour} au {date_dans_7_jours}"
df_meteo_complet['periode'] = periode_texte

In [ ]:
df_meteo_complet.to_csv("src/top_destination.csv", index=False)

In [13]:
df_meteo_complet.sort_values('beau_temps').head(5)

,id,ville,latitude,longitude,beau_temps,temperature,pluie,humidite,periode
20,21,Marseille,43.296399,5.377789,95.94,32.94,0.00,63.00,du 24/06/2026 au 01/07/2026
18,19,Bormes les Mimosas,43.150697,6.341928,97.59,34.73,0.00,62.86,du 24/06/2026 au 01/07/2026
19,20,Cassis,43.214036,5.539632,98.93,33.21,0.00,65.71,du 24/06/2026 au 01/07/2026
9,10,Chateau du Haut Koenigsbourg,48.249382,7.343941,101.26,30.79,0.04,70.43,du 24/06/2026 au 01/07/2026
5,6,Paris,48.858890,2.320041,101.94,31.97,0.26,69.71,du 24/06/2026 au 01/07/2026


### Visualisation du résultat

In [14]:

# on prend le top 5 des villes 
fig = px.scatter_mapbox(
    data_frame=df_meteo_complet.sort_values('beau_temps').head(5),  
    lat="latitude",
    lon="longitude",
    text="ville",
    color="ville",  
    zoom=4
)

# augmentation du texte des villes cibles pour les rendre plus lisible
fig.update_traces(
    textposition="top center",
    textfont=dict(size=18, color="black")  # Rouge pour que ça ressorte bien sur le fond
)

# Fond de carte OpenStreetMap gratuit (pas besoin de token Mapbox payant)
fig.update_layout(
    mapbox_style="open-street-map"
)

fig.show()

C:\Users\julie\AppData\Local\Temp\ipykernel_20772\2427403126.py:2: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
